# P2 — Clasificación del desequilibrio horario por estación

## Objetivo

Entrenar y evaluar modelos de clasificación para anticipar el nivel de desequilibrio horario de las estaciones de MiBici, utilizando información temporal y los flujos observados previamente.

## Enfoque experimental

Se utilizarán conjuntos de entrenamiento, validación y prueba separados cronológicamente para evitar una partición aleatoria que mezcle periodos temporales. Las clases de desequilibrio son provisionales y presentan desbalance, por lo que la evaluación considerará métricas por clase y no únicamente la exactitud global.

La clasificación representa el desequilibrio observado en la siguiente hora con registro disponible; no constituye todavía una medición directa del riesgo operativo real de una estación.

In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
DATA_PROCESSED = ROOT / "data" / "processed"

train_model = pd.read_parquet(DATA_PROCESSED / "p2_train.parquet")
valid_model = pd.read_parquet(DATA_PROCESSED / "p2_valid.parquet")
test_model = pd.read_parquet(DATA_PROCESSED / "p2_test.parquet")

print("Entrenamiento:", train_model.shape)
print("Validación:", valid_model.shape)
print("Prueba:", test_model.shape)

Entrenamiento: (1898168, 22)
Validación: (316712, 22)
Prueba: (315366, 22)


In [3]:
FEATURES = [
    "dia_semana",
    "mes",
    "hora_sin",
    "hora_cos",
    "flujo_neto_anterior",
    "salidas_anterior",
    "llegadas_anterior",
]

TARGET = "riesgo_siguiente"

X_train = train_model[FEATURES]
y_train = train_model[TARGET]

X_valid = valid_model[FEATURES]
y_valid = valid_model[TARGET]

X_test = test_model[FEATURES]
y_test = test_model[TARGET]

print("Variables predictoras:", FEATURES)
print("Objetivo:", TARGET)
print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_valid:", X_valid.shape)
print("Dimensiones de X_test:", X_test.shape)

Variables predictoras: ['dia_semana', 'mes', 'hora_sin', 'hora_cos', 'flujo_neto_anterior', 'salidas_anterior', 'llegadas_anterior']
Objetivo: riesgo_siguiente
Dimensiones de X_train: (1898168, 7)
Dimensiones de X_valid: (316712, 7)
Dimensiones de X_test: (315366, 7)


In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

modelo_base = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        class_weight="balanced",
        max_iter=300,
        random_state=42,
    ),
)

modelo_base.fit(X_train, y_train)

print("Modelo base entrenado correctamente.")

Modelo base entrenado correctamente.


In [5]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
)

# Predecir las clases del conjunto de validación
y_valid_pred = modelo_base.predict(X_valid)

# Métricas principales
print("Balanced accuracy:", round(
    balanced_accuracy_score(y_valid, y_valid_pred), 4
))

print("F1 macro:", round(
    f1_score(y_valid, y_valid_pred, average="macro"), 4
))

print("\nReporte de clasificación:")
print(classification_report(
    y_valid,
    y_valid_pred,
    labels=["bajo", "medio", "alto"],
    zero_division=0,
))

print("\nMatriz de confusión:")
print(confusion_matrix(
    y_valid,
    y_valid_pred,
    labels=["bajo", "medio", "alto"],
))

Balanced accuracy: 0.4355
F1 macro: 0.3595

Reporte de clasificación:
              precision    recall  f1-score   support

        bajo       0.90      0.68      0.78    275746
       medio       0.11      0.21      0.15     27598
        alto       0.10      0.42      0.16     13368

    accuracy                           0.63    316712
   macro avg       0.37      0.44      0.36    316712
weighted avg       0.80      0.63      0.70    316712


Matriz de confusión:
[[187621  42336  45789]
 [ 14528   5725   7345]
 [  5196   2575   5597]]


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

modelo_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

modelo_rf.fit(X_train, y_train)

y_valid_pred_rf = modelo_rf.predict(X_valid)

print("Balanced accuracy:",
      round(balanced_accuracy_score(y_valid, y_valid_pred_rf), 4))

print("F1 macro:",
      round(f1_score(y_valid, y_valid_pred_rf, average="macro"), 4))

print("\nReporte de clasificación:")
print(classification_report(
    y_valid,
    y_valid_pred_rf,
    labels=["bajo", "medio", "alto"],
    zero_division=0,
))

Balanced accuracy: 0.5085
F1 macro: 0.3743

Reporte de clasificación:
              precision    recall  f1-score   support

        bajo       0.94      0.61      0.74    275746
       medio       0.12      0.32      0.17     27598
        alto       0.13      0.59      0.21     13368

    accuracy                           0.59    316712
   macro avg       0.39      0.51      0.37    316712
weighted avg       0.83      0.59      0.67    316712



In [7]:
modelo_rf_v2 = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

modelo_rf_v2.fit(X_train, y_train)

y_valid_pred_rf_v2 = modelo_rf_v2.predict(X_valid)

print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_valid, y_valid_pred_rf_v2), 4)
)

print(
    "F1 macro:",
    round(f1_score(y_valid, y_valid_pred_rf_v2, average="macro"), 4)
)

print("\nReporte de clasificación:")
print(classification_report(
    y_valid,
    y_valid_pred_rf_v2,
    labels=["bajo", "medio", "alto"],
    zero_division=0,
))

Balanced accuracy: 0.4886
F1 macro: 0.3725

Reporte de clasificación:
              precision    recall  f1-score   support

        bajo       0.93      0.63      0.75    275746
       medio       0.12      0.31      0.17     27598
        alto       0.12      0.53      0.20     13368

    accuracy                           0.60    316712
   macro avg       0.39      0.49      0.37    316712
weighted avg       0.83      0.60      0.68    316712



## Comparación preliminar de modelos

Se evaluaron tres experimentos sobre el conjunto de validación: regresión logística con ponderación de clases y dos configuraciones de Random Forest.

La regresión logística obtuvo una balanced accuracy de 0.4355 y un F1 macro de 0.3595. El primer Random Forest, con 150 árboles, profundidad máxima de 15 y mínimo de 20 registros por hoja, mejoró estas métricas a 0.5085 y 0.3743, respectivamente. También elevó el recall de la clase alto de 0.42 a 0.59.

Una segunda configuración de Random Forest, con profundidad máxima de 20 y mínimo de 10 registros por hoja, obtuvo una balanced accuracy de 0.4886 y un F1 macro de 0.3725. Aunque su F1 macro fue cercano al del primer Random Forest, presentó menor capacidad de detección de la clase alto.

Por estos resultados, se conserva provisionalmente el primer Random Forest como candidato principal. La selección definitiva deberá considerar las métricas por clase y la evaluación final sobre el conjunto de prueba, que no se ha utilizado para seleccionar los modelos.

In [8]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
import pandas as pd

experimentos = [
    ("Regresión logística", modelo_base, y_valid_pred),
    ("Random Forest v1", modelo_rf, y_valid_pred_rf),
    ("Random Forest v2", modelo_rf_v2, y_valid_pred_rf_v2),
]

resultados = []

for nombre, modelo, predicciones in experimentos:
    resultados.append({
        "modelo": nombre,
        "balanced_accuracy": balanced_accuracy_score(
            y_valid, predicciones
        ),
        "f1_macro": f1_score(
            y_valid, predicciones, average="macro"
        ),
        "recall_alto": recall_score(
            y_valid, predicciones, labels=["alto"],
            average="macro", zero_division=0
        ),
        "precision_alto": precision_score(
            y_valid, predicciones, labels=["alto"],
            average="macro", zero_division=0
        ),
        "f1_alto": f1_score(
            y_valid, predicciones, labels=["alto"],
            average="macro", zero_division=0
        ),
    })

tabla_resultados = pd.DataFrame(resultados)
display(tabla_resultados.round(4))

,modelo,balanced_accuracy,f1_macro,recall_alto,precision_alto,f1_alto
0,Regresión logística,0.4355,0.3595,0.4187,0.0953,0.1553
1,Random Forest v1,0.5085,0.3743,0.5946,0.1274,0.2098
2,Random Forest v2,0.4886,0.3725,0.5252,0.1202,0.1956


In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

# Muestra reproducible para evaluar el costo y el potencial del modelo
muestra_train = train_model.sample(
    n=300_000,
    random_state=42,
)

X_train_muestra = muestra_train[FEATURES]
y_train_muestra = muestra_train[TARGET]

modelo_hgb = HistGradientBoostingClassifier(
    max_iter=100,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=42,
    class_weight="balanced",
)

modelo_hgb.fit(X_train_muestra, y_train_muestra)

y_valid_pred_hgb = modelo_hgb.predict(X_valid)

print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_valid, y_valid_pred_hgb), 4),
)
print(
    "F1 macro:",
    round(f1_score(y_valid, y_valid_pred_hgb, average="macro"), 4),
)
print("\nReporte de clasificación:")
print(classification_report(
    y_valid,
    y_valid_pred_hgb,
    labels=["bajo", "medio", "alto"],
    zero_division=0,
))

Balanced accuracy: 0.5115
F1 macro: 0.3729

Reporte de clasificación:
              precision    recall  f1-score   support

        bajo       0.94      0.62      0.74    275746
       medio       0.12      0.29      0.17     27598
        alto       0.13      0.63      0.21     13368

    accuracy                           0.59    316712
   macro avg       0.39      0.51      0.37    316712
weighted avg       0.83      0.59      0.67    316712



In [10]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

nuevo_resultado = {
    "modelo": "HistGradientBoosting (muestra)",
    "balanced_accuracy": balanced_accuracy_score(
        y_valid, y_valid_pred_hgb
    ),
    "f1_macro": f1_score(
        y_valid, y_valid_pred_hgb, average="macro"
    ),
    "recall_alto": recall_score(
        y_valid, y_valid_pred_hgb,
        labels=["alto"], average="macro", zero_division=0
    ),
    "precision_alto": precision_score(
        y_valid, y_valid_pred_hgb,
        labels=["alto"], average="macro", zero_division=0
    ),
    "f1_alto": f1_score(
        y_valid, y_valid_pred_hgb,
        labels=["alto"], average="macro", zero_division=0
    ),
}

tabla_resultados = pd.concat(
    [tabla_resultados, pd.DataFrame([nuevo_resultado])],
    ignore_index=True,
)

display(
    tabla_resultados
    .sort_values("balanced_accuracy", ascending=False)
    .round(4)
)

,modelo,balanced_accuracy,f1_macro,recall_alto,precision_alto,f1_alto
3,HistGradientBoosting (muestra),0.5115,0.3729,0.6292,0.1255,0.2092
1,Random Forest v1,0.5085,0.3743,0.5946,0.1274,0.2098
2,Random Forest v2,0.4886,0.3725,0.5252,0.1202,0.1956
0,Regresión logística,0.4355,0.3595,0.4187,0.0953,0.1553


## Conclusiones de los experimentos preliminares

Se compararon cuatro experimentos de clasificación: regresión logística, dos configuraciones de Random Forest y HistGradientBoosting entrenado con una muestra de 300,000 registros. La evaluación se realizó sobre el conjunto de validación, manteniendo separado el conjunto de prueba.

HistGradientBoosting obtuvo la mayor balanced accuracy (0.5115) y el mayor recall para la clase alto (0.6292). Por su parte, Random Forest v1 alcanzó el mayor F1 macro (0.3743), frente a 0.3729 de HistGradientBoosting. Estos resultados muestran que el modelo con mayor capacidad de detección de la clase minoritaria no necesariamente ofrece el mejor equilibrio global.

La precisión de la clase alto se mantuvo alrededor de 0.13 en ambos candidatos, lo que evidencia una cantidad considerable de falsas alarmas. Por ello, los resultados deben interpretarse como una primera aproximación a la clasificación del desequilibrio de flujo, no como una predicción validada del riesgo operativo real.

Como siguiente etapa, se deberá seleccionar el modelo según el objetivo prioritario del proyecto, realizar una comparación final bajo condiciones equivalentes cuando sea viable y evaluar el candidato elegido sobre el conjunto de prueba. Esta evaluación permitirá estimar su desempeño sobre un periodo temporal que no se utilizó para seleccionar los modelos.

In [11]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Predicción final sobre datos no utilizados para seleccionar el modelo
y_test_pred = modelo_rf.predict(X_test)

# Métricas globales
balanced_acc_test = balanced_accuracy_score(
    y_test, y_test_pred
)

f1_macro_test = f1_score(
    y_test, y_test_pred, average="macro"
)

print("EVALUACIÓN FINAL — RANDOM FOREST V1")
print("-" * 45)
print(f"Balanced accuracy: {balanced_acc_test:.4f}")
print(f"F1 macro:          {f1_macro_test:.4f}")

print("\nReporte de clasificación:")
print(classification_report(
    y_test,
    y_test_pred,
    labels=["bajo", "medio", "alto"],
    digits=4,
    zero_division=0,
))

print("Matriz de confusión:")
print(confusion_matrix(
    y_test,
    y_test_pred,
    labels=["bajo", "medio", "alto"],
))

EVALUACIÓN FINAL — RANDOM FOREST V1
---------------------------------------------
Balanced accuracy: 0.5047
F1 macro:          0.3701

Reporte de clasificación:
              precision    recall  f1-score   support

        bajo     0.9389    0.6151    0.7433    275996
       medio     0.1168    0.3136    0.1703     26637
        alto     0.1182    0.5854    0.1967     12733

    accuracy                         0.5885    315366
   macro avg     0.3913    0.5047    0.3701    315366
weighted avg     0.8364    0.5885    0.6728    315366

Matriz de confusión:
[[169775  59750  46471]
 [  9156   8354   9127]
 [  1886   3393   7454]]


## Evaluación final y conclusiones de P2

Se seleccionó provisionalmente Random Forest v1 como candidato principal, priorizando el F1 macro y utilizando balanced accuracy como métrica complementaria. En validación, el modelo obtuvo un F1 macro de 0.3743 y una balanced accuracy de 0.5085. En la evaluación final sobre el conjunto de prueba, alcanzó un F1 macro de 0.3701 y una balanced accuracy de 0.5047.

La diferencia reducida entre validación y prueba sugiere que el desempeño se mantiene relativamente estable en el periodo temporal evaluado. Sin embargo, el rendimiento global continúa siendo limitado, especialmente para las clases medio y alto.

En la clase alto, el modelo obtuvo un recall de 0.5854, identificando 7,454 de los 12,733 casos reales. No obstante, su precisión fue de 0.1182, lo que evidencia una cantidad considerable de falsas alarmas. La clase medio también presentó un F1 bajo, de 0.1703. Por ello, aunque el modelo ofrece una primera aproximación reproducible, no se considera suficiente para su uso operativo sin mejoras adicionales.

Como siguientes pasos, se propone investigar variables adicionales, evaluar estrategias para mejorar la clasificación de las clases minoritarias y comparar modelos bajo condiciones equivalentes. También será necesario revisar la definición de las etiquetas y la cobertura de los registros horarios. Las categorías actuales representan desequilibrios de flujo observados y no una medición directa del inventario ni del riesgo operativo real de las estaciones.

En conclusión, el experimento establece una línea base cuantitativa para futuras iteraciones. La evaluación se realizó sobre un periodo temporal reservado para prueba, sin utilizar sus resultados para ajustar el modelo seleccionado.

In [12]:
import mlflow

print("Versión de MLflow:", mlflow.__version__)

Versión de MLflow: 3.17.1


/opt/anaconda3/envs/mibici/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
from pathlib import Path
import mlflow

# Directorio para almacenar los resultados de MLflow
MLFLOW_DIR = Path("../mlruns").resolve()
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)

# Base de datos local para el seguimiento de experimentos
MLFLOW_DB = MLFLOW_DIR / "mlflow.db"
MLFLOW_TRACKING_URI = f"sqlite:///{MLFLOW_DB}"

# Configurar MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Crear o recuperar el experimento de P2
EXPERIMENT_NAME = "MiBici_P2_Clasificacion_Riesgo"

mlflow.set_experiment(EXPERIMENT_NAME)

print("Experimento:", EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Directorio:", MLFLOW_DIR)

2026/10/17 23:56:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/17 23:56:42 INFO mlflow.store.db.utils: Updating database tables
2026/10/17 23:56:43 INFO mlflow.tracking.fluent: Experiment with name 'MiBici_P2_Clasificacion_Riesgo' does not exist. Creating a new experiment.


Experimento: MiBici_P2_Clasificacion_Riesgo
Tracking URI: sqlite:////Users/edgargonzalezpaz/Documents/MLOps/TC5061-Proyecto-Equipo37/TC5061-Proyecto-Equipo37/mlruns/mlflow.db
Directorio: /Users/edgargonzalezpaz/Documents/MLOps/TC5061-Proyecto-Equipo37/TC5061-Proyecto-Equipo37/mlruns


In [15]:
import mlflow
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Registrar el modelo seleccionado y sus resultados en test
with mlflow.start_run(run_name="RandomForest_v1_test"):

    # Parámetros del modelo
    mlflow.log_params(modelo_rf.get_params())

    # Métricas globales
    mlflow.log_metric(
        "balanced_accuracy_test",
        float(balanced_accuracy_score(y_test, y_test_pred))
    )

    mlflow.log_metric(
        "f1_macro_test",
        float(f1_score(y_test, y_test_pred, average="macro"))
    )

    # Métricas individuales por clase
    reporte = classification_report(
        y_test,
        y_test_pred,
        output_dict=True,
        zero_division=0
    )

    for clase, metricas in reporte.items():
        if isinstance(metricas, dict):
            for metrica, valor in metricas.items():
                mlflow.log_metric(
                    f"{clase}_{metrica}",
                    float(valor)
                )

    # Matriz de confusión como artefacto
    cm = confusion_matrix(y_test, y_test_pred)
    ruta_cm = "matriz_confusion_test.csv"

    import pandas as pd
    pd.DataFrame(cm).to_csv(ruta_cm, index=False)
    mlflow.log_artifact(ruta_cm)

    print("Experimento registrado correctamente.")
    print("Balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred))
    print("F1 macro:", f1_score(y_test, y_test_pred, average="macro"))

Experimento registrado correctamente.
Balanced accuracy: 0.5047225427248373
F1 macro: 0.37009111625034224


In [16]:
import mlflow
from sklearn.metrics import balanced_accuracy_score, f1_score

experimentos = [
    ("RandomForest_v1_validacion", y_valid, y_valid_pred_rf),
    ("RandomForest_v2_validacion", y_valid, y_valid_pred_rf_v2),
    ("HistGradientBoosting_validacion", y_valid, y_valid_pred_hgb),
]

for nombre, y_real, y_pred in experimentos:
    with mlflow.start_run(run_name=nombre):
        mlflow.log_metric(
            "balanced_accuracy_valid",
            float(balanced_accuracy_score(y_real, y_pred))
        )
        mlflow.log_metric(
            "f1_macro_valid",
            float(f1_score(y_real, y_pred, average="macro"))
        )

        print(
            nombre,
            "| Balanced accuracy:",
            round(balanced_accuracy_score(y_real, y_pred), 4),
            "| F1 macro:",
            round(f1_score(y_real, y_pred, average="macro"), 4)
        )

RandomForest_v1_validacion | Balanced accuracy: 0.5085 | F1 macro: 0.3743
RandomForest_v2_validacion | Balanced accuracy: 0.4886 | F1 macro: 0.3725
HistGradientBoosting_validacion | Balanced accuracy: 0.5115 | F1 macro: 0.3729


In [17]:
experimento = mlflow.get_experiment_by_name(
    "MiBici_P2_Clasificacion_Riesgo"
)

runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    output_format="pandas"
)

columnas = [
    "tags.mlflow.runName",
    "metrics.balanced_accuracy_valid",
    "metrics.f1_macro_valid",
    "metrics.balanced_accuracy_test",
    "metrics.f1_macro_test",
]

display(
    runs[[c for c in columnas if c in runs.columns]]
    .sort_values(
        "metrics.f1_macro_valid",
        ascending=False,
        na_position="last"
    )
)

,tags.mlflow.runName,metrics.balanced_accuracy_valid,metrics.f1_macro_valid,metrics.balanced_accuracy_test,metrics.f1_macro_test
2,RandomForest_v1_validacion,0.508512,0.374298,NaN,NaN
0,HistGradientBoosting_validacion,0.511529,0.372928,NaN,NaN
1,RandomForest_v2_validacion,0.488564,0.372506,NaN,NaN
3,RandomForest_v1_test,NaN,NaN,0.504723,0.370091


In [18]:
runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    output_format="pandas"
)

columnas_parametros = [
    c for c in runs.columns
    if c.startswith("params.")
]

display(
    runs[["tags.mlflow.runName"] + columnas_parametros]
)

,tags.mlflow.runName,params.max_depth,params.monotonic_cst,params.max_features,params.min_weight_fraction_leaf,params.n_estimators,params.verbose,params.oob_score,params.criterion,params.random_state,params.warm_start,params.min_impurity_decrease,params.ccp_alpha,params.class_weight,params.min_samples_leaf,params.min_samples_split,params.bootstrap,params.max_samples,params.max_leaf_nodes,params.n_jobs
0,HistGradientBoosting_validacion,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,RandomForest_v2_validacion,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,RandomForest_v1_validacion,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,RandomForest_v1_test,15,None,sqrt,0.0,150,0,False,gini,42,False,0.0,0.0,balanced,20,2,True,None,None,-1


In [19]:
# Asociar cada ejecución con su modelo
modelos_validacion = {
    "RandomForest_v1_validacion": modelo_rf,
    "RandomForest_v2_validacion": modelo_rf_v2,
    "HistGradientBoosting_validacion": modelo_hgb,
}

# Recuperar el experimento existente
experimento = mlflow.get_experiment_by_name(
    "MiBici_P2_Clasificacion_Riesgo"
)

# Completar los parámetros en las ejecuciones existentes
for nombre, modelo_actual in modelos_validacion.items():

    runs_modelo = mlflow.search_runs(
        experiment_ids=[experimento.experiment_id],
        filter_string=f"tags.mlflow.runName = '{nombre}'",
        output_format="pandas"
    )

    if runs_modelo.empty:
        print(f"No se encontró la ejecución: {nombre}")
        continue

    run_id = runs_modelo.iloc[0]["run_id"]

    with mlflow.start_run(run_id=run_id):
        mlflow.log_params(modelo_actual.get_params())

    print(f"Parámetros registrados: {nombre}")

Parámetros registrados: RandomForest_v1_validacion
Parámetros registrados: RandomForest_v2_validacion
Parámetros registrados: HistGradientBoosting_validacion


In [20]:
import json
import tempfile
from pathlib import Path

import mlflow
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

experimento = mlflow.get_experiment_by_name(
    "MiBici_P2_Clasificacion_Riesgo"
)

evaluaciones = [
    ("RandomForest_v1_validacion", y_valid, y_valid_pred_rf),
    ("RandomForest_v2_validacion", y_valid, y_valid_pred_rf_v2),
    ("HistGradientBoosting_validacion", y_valid, y_valid_pred_hgb),
    ("RandomForest_v1_test", y_test, y_test_pred),
]

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)

    for nombre, y_real, y_pred in evaluaciones:
        runs = mlflow.search_runs(
            experiment_ids=[experimento.experiment_id],
            filter_string=f"tags.mlflow.runName = '{nombre}'",
            output_format="pandas"
        )

        if runs.empty:
            print(f"No se encontró: {nombre}")
            continue

        carpeta = tmpdir / nombre
        carpeta.mkdir()

        reporte = classification_report(
            y_real, y_pred, output_dict=True, zero_division=0
        )

        with open(carpeta / "classification_report.json", "w") as f:
            json.dump(reporte, f, indent=2)

        pd.DataFrame(
            confusion_matrix(y_real, y_pred)
        ).to_csv(carpeta / "confusion_matrix.csv", index=False)

        with mlflow.start_run(run_id=runs.iloc[0]["run_id"]):
            mlflow.log_artifacts(str(carpeta), artifact_path="evaluation")

        print(f"Artefactos registrados: {nombre}")

Artefactos registrados: RandomForest_v1_validacion
Artefactos registrados: RandomForest_v2_validacion
Artefactos registrados: HistGradientBoosting_validacion
Artefactos registrados: RandomForest_v1_test


### Seguimiento de experimentos con MLflow

Se implementó MLflow para registrar y consultar los experimentos de clasificación de riesgo de MiBici. Se configuró un almacenamiento local mediante SQLite y se creó el experimento `MiBici_P2_Clasificacion_Riesgo`.

Se registraron tres ejecuciones de validación: Random Forest v1, Random Forest v2 e HistGradientBoosting. Para estas ejecuciones se almacenaron los parámetros de los modelos, las métricas de *balanced accuracy* y F1 macro, así como los reportes de clasificación y las matrices de confusión. También se registró la evaluación final de Random Forest v1 sobre el conjunto de prueba (*test*).

Random Forest v1 obtuvo el mejor F1 macro en validación (0.3743), mientras que HistGradientBoosting alcanzó la mayor *balanced accuracy* (0.5115). Se mantuvo Random Forest v1 como candidato seleccionado debido a la prioridad establecida de equilibrar el desempeño entre clases. En test obtuvo una *balanced accuracy* de 0.5047 y un F1 macro de 0.3701.

Este flujo facilita la trazabilidad de los experimentos y la consulta de sus resultados sin necesidad de repetir el entrenamiento. Como mejora futura, se propone registrar las versiones de los datos y del código, además de automatizar la ejecución completa del pipeline. La comparación debe interpretarse considerando que HistGradientBoosting se entrenó con una muestra de datos.


In [21]:
import mlflow
from pathlib import Path

# Configurar nuevamente el seguimiento local
MLFLOW_DB = Path("../mlruns/mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

# Consultar los experimentos registrados
experimento = mlflow.get_experiment_by_name(
    "MiBici_P2_Clasificacion_Riesgo"
)

runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    output_format="pandas"
)

columnas = [
    "tags.mlflow.runName",
    "metrics.balanced_accuracy_valid",
    "metrics.f1_macro_valid",
    "metrics.balanced_accuracy_test",
    "metrics.f1_macro_test"
]

display(
    runs[[c for c in columnas if c in runs.columns]]
    .sort_values("tags.mlflow.runName")
)

,tags.mlflow.runName,metrics.balanced_accuracy_valid,metrics.f1_macro_valid,metrics.balanced_accuracy_test,metrics.f1_macro_test
0,HistGradientBoosting_validacion,0.511529,0.372928,NaN,NaN
3,RandomForest_v1_test,NaN,NaN,0.504723,0.370091
2,RandomForest_v1_validacion,0.508512,0.374298,NaN,NaN
1,RandomForest_v2_validacion,0.488564,0.372506,NaN,NaN


In [22]:
import mlflow
from pathlib import Path

# Configurar el almacenamiento local de MLflow
MLFLOW_DB = Path("../mlruns/mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

# Recuperar el experimento
experimento = mlflow.get_experiment_by_name(
    "MiBici_P2_Clasificacion_Riesgo"
)

if experimento is None:
    print("No se encontró el experimento.")
else:
    runs = mlflow.search_runs(
        experiment_ids=[experimento.experiment_id],
        output_format="pandas"
    )

    # Revisar los artefactos de cada ejecución
    for _, run in runs.iterrows():
        nombre = run["tags.mlflow.runName"]
        run_id = run["run_id"]

        print(f"\nExperimento: {nombre}")

        artefactos = mlflow.artifacts.list_artifacts(run_id)

        for artefacto in artefactos:
            print(
                f"  - {artefacto.path}/"
                if artefacto.is_dir
                else f"  - {artefacto.path}"
            )


Experimento: HistGradientBoosting_validacion

Experimento: RandomForest_v2_validacion

Experimento: RandomForest_v1_validacion

Experimento: RandomForest_v1_test


In [24]:
import mlflow
from pathlib import Path

MLFLOW_DB = Path("../mlruns/mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")

experimento = mlflow.get_experiment_by_name(
    "MiBici_P2_Clasificacion_Riesgo"
)

runs = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    output_format="pandas"
)

for _, run in runs.iterrows():
    nombre = run["tags.mlflow.runName"]
    run_id = run["run_id"]

    print(f"\n{nombre}")

    def listar_artefactos(run_id, ruta=""):
        elementos = mlflow.artifacts.list_artifacts(
            run_id=run_id,
            artifact_path=ruta if ruta else None
        )

        for item in elementos:
            print(f"  {'[CARPETA]' if item.is_dir else '[ARCHIVO]'} {item.path}")

            if item.is_dir:
                listar_artefactos(run_id, item.path)

    listar_artefactos(run_id)


HistGradientBoosting_validacion
  [CARPETA] evaluation
  [ARCHIVO] evaluation/classification_report.json
  [ARCHIVO] evaluation/confusion_matrix.csv

RandomForest_v2_validacion
  [CARPETA] evaluation
  [ARCHIVO] evaluation/classification_report.json
  [ARCHIVO] evaluation/confusion_matrix.csv

RandomForest_v1_validacion
  [CARPETA] evaluation
  [ARCHIVO] evaluation/classification_report.json
  [ARCHIVO] evaluation/confusion_matrix.csv

RandomForest_v1_test
  [CARPETA] evaluation
  [ARCHIVO] evaluation/classification_report.json
  [ARCHIVO] evaluation/confusion_matrix.csv
  [ARCHIVO] matriz_confusion_test.csv


### Seguimiento de experimentos con MLflow

Se implementó MLflow para registrar y consultar los experimentos de clasificación de riesgo de MiBici. Se configuró un almacenamiento local mediante SQLite y se creó el experimento `MiBici_P2_Clasificacion_Riesgo`.

Se registraron tres ejecuciones de validación: Random Forest v1, Random Forest v2 e HistGradientBoosting. Para estas ejecuciones se almacenaron los parámetros de los modelos, las métricas de *balanced accuracy* y F1 macro, así como los reportes de clasificación y las matrices de confusión. También se registró la evaluación final de Random Forest v1 sobre el conjunto de prueba (*test*).

Random Forest v1 obtuvo el mejor F1 macro en validación (0.3743), mientras que HistGradientBoosting alcanzó la mayor *balanced accuracy* (0.5115). Se mantuvo Random Forest v1 como candidato seleccionado debido a la prioridad establecida de equilibrar el desempeño entre clases. En test obtuvo una *balanced accuracy* de 0.5047 y un F1 macro de 0.3701.

Los resultados de test muestran una ligera disminución respecto a validación, lo que sugiere estabilidad temporal de las métricas globales. Sin embargo, el desempeño en las clases minoritarias sigue siendo limitado, especialmente en precisión. La comparación también debe interpretarse considerando que HistGradientBoosting se entrenó con una muestra de datos.

Como mejora futura, se propone registrar las versiones de los datos y del código, además de automatizar la ejecución completa del pipeline. Este flujo permite consultar y comparar experimentos sin necesidad de repetir el entrenamiento.
